### SOLAR POWER GENERATION PREDICTION MODEL 
    --- Predict solar power generation for a given period of time for either a real-life plant or a hypothetical new plant in a region.
    --- The model should understand the universal relationship between weather/sunlight and power generation so it can be applied to different
        locations


### SolarPlantData1 : SolarPlant Power Generation Monitoring (2023-2025)
    --- KAGGLE LINK: https://www.kaggle.com/datasets/juanschafle/solarplant-power-generation-monitoring-2023-2025
    --- RAW data: Raw Data/solar_plant_generation_dataset.csv
    --- Notes: This is a great dataset due to much longer time coverage and it has better weather data. Most notable the cloud coverage percentage
               which will come in really handy for prediction during non sunny weathers. It has only one data for generation which I believeis enough 
               as it lines up with daily_yeild. So far the table has the following:
               - Irridance: the amount solar power recieved by the panels
               - Module_temperature: the temp of the panels
               - Ambient_temperature: the temp of the environment
               - daily_yeild/generation power: amount of power recieved up to that point for the day
               - timestamp, cloud_coverage
               - Plant_ID: a fake id added and numbered in the format of the other dataset for joining purposed later on
               
               This table for the most part is good in terms of column and missing/ NaN values. What it needs is to be expanded to 15 minutes using
               linear interpolation to match the incrementing of SolarPlantData2.ipynb. This will be a better method as shrinking the other one to
               1hr increment means we lose valuable data. 

### SolarPlantData2 : Solar Power Generation Data
    --- KAGGLE LINK: https://www.kaggle.com/datasets/anikannal/solar-power-generation-data
    --- RAW data: Raw Data/Plant_Generation_&_Weather_censor_Data
    --- Notes: This is our best dataset, it has both weather and generation data. SolarPlantData2.ipynb - this notebook combines all the tables into  
               one table in preperation to be connectied to SolarPlantData1.ipynb. So far the combined table has the following:
               - Irridance: the amount solar power recieved by the panels
               - Module_temperature: the temp of the panels
               - Ambient_temperature: the temp of the environment
               - daily_yeild/generation power: amount of power recieved up to that point for the day
               - timestamp, plant id
               - source key, ac_power, dc_power taken out. AC and DC power measure the power at the exact momment and not cummulative so they are not
               as valueable as cloud coverage is.
               
               This table for the most part has all the column needed only. The null values/ NaNs have been taken out. I have also combined all panels 
               to get the power generation total for each time increment. The only key column that is missing is the cloud coverage the other dataset 
               has. The solution is to either drop, which is difficult to drop as it is key information or use a weather API, to fill out the the 
               cloud coverage column based on the location and the date.
               

### Final Table:
    --- This will be a third ipynb that will combined the completed SolarPlantData1 and SolarPlantData2 and the table we will use for our algorithm
        and our model. Before joinining we need to:
        - expand and correct the incrementation for SolarPlantData1 to 15 min
        - add the cloud coverage data to SolarPlantData2 using weather API
        - *** MAKE THE UNITS CONSISTENT BECAUSE AS OF NOW THEY ARE NOT ***
        - then in the new notenook:
          - reorder the column
          - make sure they have same column names
          - combine
          - check if all the values are correct

================================================================================================================================================================================================================================================================================================================================================================================================================================================================================================

# MODEL TRAINING TEAM

### Column rundown
* IRRADIATION(W/m²) - real sensor reading of sunlight intensity, one of the core inputs.
* WIND_SPEED(m/s) - pulled in afterward from a weather API for the plant's estimated location, used in the physics temperature calculation.
* CLOUD_COVER(%) - also pulled from the weather API, 0 means clear sky and 100 means fully overcast.
* AMBIENT_TEMPERATURE(°C) - real sensor reading, core input.
* THEORETICAL_MODULE_TEMPERATURE(°C) - the physics formula's estimate of panel temperature, calculated from ambient temperature, irradiation, and wind speed.
* ACTUAL_MODULE_TEMPERATURE(°C) - real sensor reading of panel temperature, kept only to compare against the theoretical estimate, not used as an input.
* THEORETICAL_DC_POWER(kW) and THEORETICAL_AC_POWER(kW) - the physics-only predicted output, scaled up using the assumed plant capacity.
* ACTUAL_DC_POWER(kW) and ACTUAL_AC_POWER(kW) - real measured output, this is the ground truth.
* DC_EFFICIENCY and AC_EFFICIENCY - real output expressed as a percentage of the single highest output ever seen in the dataset, this is not a true efficiency number, just a relative level, and should be explained that way if it comes up.
* INVERTOR_EFFICIENCY - real measured AC power divided by real measured DC power on the same row, fine to look at for analysis but should not be fed into the physics baseline since it is built from real output.
* DAILY_YIELD(kWh) and TOTAL_YIELD(kWh) - real cumulative energy counters, not useful as model inputs since they are running totals rather than a snapshot of conditions.
* THEORETICAL_DC_PER_KWP and THEORETICAL_AC_PER_KWP - the physics baseline expressed per kWp of capacity instead of an absolute number, this is the size-independent version to build from.
* CALCULATED_PLANT_CAPACITY(kWp) - one single fixed assumed capacity applied to every row, this is a stated assumption, not something pulled from any power reading.
* DC_RESIDUAL(kW) and AC_RESIDUAL(kW) - actual output minus theoretical output, in absolute kW, this needs to be divided by capacity before training, not used directly.

#### The variables that are not columns in this table but exist in the physicsCalc.py and will later become optional user inputs in the app, each with a typical default value if the user does not provide one: panel tilt angle, panel azimuth angle, inverter efficiency rating, panel temperature coefficient, and system degradation rate. These do not need to be touched by the training team, they are listed here so it is clear the physics formulas were written to accept overrides for them.


### What the model is training with and for
* The model takes irradiation, ambient temperature, wind speed, cloud cover, and a set of time-of-day and time-of-year features, and learns to predict the per kWp gap between what the physics formula expects and what the real plant actually produced. We are predicting this gap instead of raw power is that raw power only makes sense for one specific plant's size, while the gap, once divided by capacity, can be scaled to any system size later.
* The algorithm to use is XGBoost. Choose this over Random Forest because it picks up on time-based patterns like hour of day and season more effectively.
* The output of the model is a single predicted number per row, the per kWp correction to add on top of the physics baseline.
* Testing should include a standard 80/20 train and test split as a first check.
* Testing should also include 5 fold cross validation across the whole dataset, reporting the score from each fold along with the average and spread, to confirm the result is not just a lucky split.
* A small set of physics based testing should be run on the combined output, specifically confirming zero irradiation always produces zero output, and confirming a cooler panel produces more output than a hotter one under the same irradiation.
* A backtest should be run against real held out data, plotting the prediction error to confirm it is scattered randomly around zero rather than consistently high or low in one direction.
* The final accuracy numbers from this backtest, plus the cross validation results, need to be written down clearly since they get shown to end users later as a confidence indicator, so they need to be accurate rather than optimistic.


### What not to use
* Do not use ACTUAL_DC_POWER, ACTUAL_AC_POWER, DC_EFFICIENCY, AC_EFFICIENCY, INVERTOR_EFFICIENCY, DAILY_YIELD, TOTAL_YIELD, or ACTUAL_MODULE_TEMPERATURE as inputs, all of these either directly contain the answer being predicted or were built from it.
* Do not use any of the THEORETICAL columns as inputs either, they are the baseline the model is correcting, not something to feed back into itself.
* Do not train on the residual in raw kW, always convert to per kWp first using the fixed capacity value.


### Limitations
* The data covers only 34 days from mid May to mid June in one region, so the model has no knowledge of monsoon, winter, or other climates, and will be increasingly unreliable the further real conditions drift from that window.
* The assumed plant capacity is a stated guess rather than a confirmed number, and since the training target is built relative to that capacity, any error in it biases every prediction the model makes proportionally.
* All of the real world imperfections the model is learning to correct for come from one specific plant's hardware and condition, and may not reflect solar installations generally.
* There is no second, independent real plant to validate against, so there is no way to confirm the model generalizes beyond the one it was trained on.


# APPLICATION TEAM

### Column rundown
* IRRADIATION(W/m²) - real sensor reading of sunlight intensity, one of the core inputs.
* WIND_SPEED(m/s) - pulled in afterward from a weather API for the plant's estimated location, used in the physics temperature calculation.
* CLOUD_COVER(%) - also pulled from the weather API, 0 means clear sky and 100 means fully overcast.
* AMBIENT_TEMPERATURE(°C) - real sensor reading, core input.
* THEORETICAL_MODULE_TEMPERATURE(°C) - the physics formula's estimate of panel temperature, calculated from ambient temperature, irradiation, and wind speed.
* ACTUAL_MODULE_TEMPERATURE(°C) - real sensor reading of panel temperature, kept only to compare against the theoretical estimate, not used as an input.
* THEORETICAL_DC_POWER(kW) and THEORETICAL_AC_POWER(kW) - the physics-only predicted output, scaled up using the assumed plant capacity.
* ACTUAL_DC_POWER(kW) and ACTUAL_AC_POWER(kW) - real measured output, this is the ground truth.
* DC_EFFICIENCY and AC_EFFICIENCY - real output expressed as a percentage of the single highest output ever seen in the dataset, this is not a true efficiency number, just a relative level, and should be explained that way if it comes up.
* INVERTOR_EFFICIENCY - real measured AC power divided by real measured DC power on the same row, fine to look at for analysis but should not be fed into the physics baseline since it is built from real output.
* DAILY_YIELD(kWh) and TOTAL_YIELD(kWh) - real cumulative energy counters, not useful as model inputs since they are running totals rather than a snapshot of conditions.
* THEORETICAL_DC_PER_KWP and THEORETICAL_AC_PER_KWP - the physics baseline expressed per kWp of capacity instead of an absolute number, this is the size-independent version to build from.
* CALCULATED_PLANT_CAPACITY(kWp) - one single fixed assumed capacity applied to every row, this is a stated assumption, not something pulled from any power reading.
* DC_RESIDUAL(kW) and AC_RESIDUAL(kW) - actual output minus theoretical output, in absolute kW, this needs to be divided by capacity before training, not used directly.

### Inputs the app should collect from the user
* Location, either typed in and geocoded or dropped as a pin on a map, this cannot be left blank, there is no reasonable default for where someone's system is.
* System capacity in kWp, this also cannot be left blank, there is no reasonable default for how large someone's system is.
* Panel tilt angle, this can be left blank, if it is, default to a value roughly equal to the location's latitude.
* Panel azimuth angle, this can be left blank, if it is, default to a typical south facing assumption or the standard orientation for the hemisphere the location is in.
* Inverter efficiency rating, this can be left blank, if it is, default to a typical value around ninety seven percent.
* Panel temperature coefficient, this can be left blank, if it is, default to a typical crystalline silicon value.
* System age or degradation rate, this can be left blank, if it is, default to treating the system as new with no degradation applied.
* Time range for the estimate, this cannot be left blank, the user needs to choose how far out they want the estimate for, anywhere from 1 up to 10 days will use real short term forecast data, and anything longer than that, up to a chosen horizon such as 10 years, will be built from historical weather instead since no service can provide real forecasted weather that far ahead.

### How to use the inputs, model, and physics together
1. Location handling - the location the user provides needs to be converted into a latitude and longitude if it was not already given as coordinates, since both the weather API call and the physics calculation depend on having real coordinates.
2. Weather retrieval - once coordinates exist, check the time range the user chose, if it falls within the 1 to 10 day short term window, call the forecast weather API directly for those exact dates, if it is longer than that, call the historical archive weather API instead, pulling as many past years of data as are reasonably available for that location so the longer range estimate is built from real historical patterns rather than a guess.
3. Physics baseline calculation - feed the weather data, along with whichever specs the user provided or the defaults filling in for the ones they did not, into the shared physics function, this produces the theoretical per kWp output for every time interval in the requested range.
4. Feature preparation for the model - build the same time based features the model was trained on, meaning sine and cosine versions of the hour and month, from the timestamps returned by the weather API.
5. Model correction - pass the weather values and time features into the trained model to get the predicted per kWp correction for each time interval.
6. Combining and scaling - add the physics output and the model correction together, then multiply by the user's entered capacity, this produces the actual expected output in real kW for their specific system.
7. Aggregation - sum or average the resulting values as needed to build daily, monthly, and yearly totals depending on which graph or table is being generated.
8. Risk range calculation - for longer projections, repeat the weather retrieval and calculation across many historical years, then calculate the spread across those years to produce the P10 through P90 style range rather than a single number.

### Non calculative features, comments, and things to tell the user
* Confidence labeling - the app should track how many optional fields the user left blank versus filled in, and show a simple confidence label or note reflecting that, more filled in fields should read as a more confident estimate.
* Out of range warning - the app should compare the requested location and season against the known range the model was actually trained on, and if it is far outside that range, show a plain note that the estimate is less reliable for conditions very different from the training data.
* Assumption disclosure - whenever a default value was used because the user left a field blank, the app should say so directly near the output, rather than presenting the number as if everything were fully specified.
* Simple tier commentary - short plain language notes next to the simple graphs, for example noting that output is expected to be lower in certain months, or translating the yearly number into something relatable like months of household power.
* Advanced tier commentary - more technical notes next to the detailed tables, for example explaining the gap between the P90 and P50 numbers in terms relevant to financial planning.
* Model versus weather uncertainty note - the app should make clear in writing that the P10 through P90 range reflects weather variability, while a separate, smaller note should show the model's own backtested error range, since these are two different kinds of uncertainty and should not be blended into one number.

### Condensed flowchart of how the app should run
User enters location and capacity, optionally more detail -> app converts location to coordinates -> app calls the weather API, short term or historical depending on the request length -> app runs the physics function on that weather data plus the user's specs or defaults -> app runs the trained model to get the correction values -> app combines physics and model output, then scales by the user's capacity -> app aggregates the results into the needed time periods -> app calculates the risk range from multiple historical years if it is a longer projection -> app checks how many defaults were used and how far the request is from the training data's range - app renders the simple view and the advanced view from the same underlying numbers, along with the confidence notes and any warnings -> user can toggle between the two views or download the full table.

### Features the user should see in the app
* Headline number - a single large number at the top showing estimated yearly output, shown in plain kWh along with a relatable comparison, this is the first thing a simple tier user sees.
* Location, needed to pull weather data and to run the physics calculation, this cannot be left blank.
* System capacity in kWp, needed to scale the physics and model output from a per kWp number into the user's real expected output, this cannot be left blank.
* Panel tilt angle, used by the physics formula to calculate how much sunlight actually hits the panel surface, this can be left blank, if it is, default to a value roughly equal to the location's latitude.
* Panel azimuth angle, used by the physics formula alongside tilt to calculate sunlight exposure, this can be left blank, if it is, default to a typical south facing assumption or the standard orientation for the hemisphere the location is in.
* Inverter efficiency rating, used by the physics formula when converting the calculated DC output into AC output, this can be left blank, if it is, default to a typical value around ninety seven percent.
* Panel temperature coefficient, used by the physics formula to calculate how much output drops as the panel heats up, this can be left blank, if it is, default to a typical crystalline silicon value.
* System age or degradation rate, used to adjust the long term output projection downward over time to reflect panel aging, this can be left blank, if it is, default to treating the system as new with no degradation applied.
* Time range selector - a control letting the user pick how far ahead they want the estimate for, with a clear label distinguishing the short term forecast based range from the longer historical pattern based range, so the user understands which kind of estimate they are looking at.
* Monthly bar chart - a bar for each month of the year showing expected output, letting the user see the seasonal shape at a glance.
* Day in the life curve - a single sample day's generation curve, rising in the morning, peaking midday, and falling in the evening, giving an intuitive visual of how a typical day behaves.
* Risk range gauge - a simple visual showing the gap between the worst realistic case and the typical case, labeled in plain words rather than statistical terms for the simple tier.
* Quantile table - a detailed table for the advanced tier showing the full P10 through P90 breakdown, by month and by year.
* Multiple day type curves - separate generation curves for a clear day, a cloudy day, and a rainy or monsoon like day, so the advanced user can see the range of daily behavior rather than just one example.
* Model confidence panel - a small section showing the model's own backtested error range, kept visually separate from the weather risk range so the two types of uncertainty are not confused.
* Degradation projection chart - a chart comparing year one output against a later year, such as year ten, adjusted for panel aging, shown only if the user provided a degradation rate.
* Transparency table - a full expandable table of every underlying value used to build the outputs, timestamp, weather inputs, physics number, model correction, and final number, available to any user who wants to dig deeper.
* Downloadable export - a button letting the user download the underlying data table as a file for their own use.
* Toggle between simple and advanced view - a single switch that changes how much detail is shown without changing the underlying numbers.
* Toggle between time periods - the ability to switch the charts and tables between daily, monthly, and yearly views.
* Suggestions panel - short actionable notes tied to the user's own numbers, for example suggesting appliance timing for a simple tier user or noting financial planning considerations for an advanced tier user.
* Risk and caveats panel - a dedicated space listing anything relevant to trustworthiness, including the out of range warning, the list of which fields were defaulted, and a plain explanation of what P50 and P90 mean.
* Input summary tab - a tab or panel showing back exactly what the user entered and what was defaulted, so they can review and correct it without having to redo the whole input process.
* Comparison tab - an optional tab letting the user compare two different input scenarios side by side, for example comparing their system at two different capacities or two different locations.
